# User-Facing Fraud Detection Model (PaySim Schema)

This notebook implements an end-to-end Machine Learning pipeline for fraud detection using realistic, user-interpretable transaction features:
- `type`: Transaction type (CASH-OUT, TRANSFER, PAYMENT, CASH-IN, DEBIT)
- `amount`: Transaction amount ($)
- `oldbalanceOrg`: Initial balance of sender before transaction
- `newbalanceOrig`: Final balance of sender after transaction
- `oldbalanceDest`: Initial balance of recipient before transaction
- `newbalanceDest`: Final balance of recipient after transaction

Unlike anonymized PCA features (V1-V28), these attributes are human-interpretable and suitable for user-facing applications.

In [1]:
import pickle, warnings, os
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
np.random.seed(42)

## 1. Dataset Generation (PaySim Schema)
Simulating 200,000 transactions with 0.15% fraud rate reflecting mobile money fraud dynamics.

In [2]:
N = 200_000
FRAUD_RATE = 0.0015
rng = np.random.default_rng(42)
TYPES = ['CASH-OUT', 'TRANSFER', 'PAYMENT', 'CASH-IN', 'DEBIT']
TPROBS = [0.35, 0.08, 0.34, 0.22, 0.01]

n_genuine = int(N * (1 - FRAUD_RATE))
n_fraud = N - n_genuine

def gen_genuine(n):
    types = rng.choice(TYPES, size=n, p=TPROBS)
    amount = np.abs(rng.exponential(scale=180_000, size=n))
    amount = np.clip(amount, 1, 10_000_000)
    oldbalOrg = np.abs(rng.exponential(scale=800_000, size=n))
    newbalOrg = np.maximum(oldbalOrg - amount, 0)
    is_payment = np.isin(types, ['PAYMENT', 'CASH-IN'])
    newbalOrg = np.where(is_payment, oldbalOrg, newbalOrg)
    oldbalDest = np.abs(rng.exponential(scale=600_000, size=n))
    newbalDest = oldbalDest + amount
    return pd.DataFrame({
        'type': types,
        'amount': np.round(amount, 2),
        'oldbalanceOrg': np.round(oldbalOrg, 2),
        'newbalanceOrig': np.round(newbalOrg, 2),
        'oldbalanceDest': np.round(oldbalDest, 2),
        'newbalanceDest': np.round(newbalDest, 2),
        'isFraud': 0,
    })

def gen_fraud(n):
    types = rng.choice(['CASH-OUT', 'TRANSFER'], size=n, p=[0.5, 0.5])
    amount = np.abs(rng.exponential(scale=500_000, size=n))
    amount = np.clip(amount, 100, 10_000_000)
    drain_fully = rng.random(n) < 0.80
    oldbalOrg = amount + rng.exponential(scale=5_000, size=n)
    newbalOrg = np.where(drain_fully, 0, oldbalOrg - amount * rng.uniform(0.5, 0.9, n))
    newbalOrg = np.maximum(newbalOrg, 0)
    dest_unchanged = rng.random(n) < 0.70
    oldbalDest = np.abs(rng.exponential(scale=10_000, size=n))
    newbalDest = np.where(dest_unchanged, oldbalDest, oldbalDest + amount)
    return pd.DataFrame({
        'type': types,
        'amount': np.round(amount, 2),
        'oldbalanceOrg': np.round(oldbalOrg, 2),
        'newbalanceOrig': np.round(newbalOrg, 2),
        'oldbalanceDest': np.round(oldbalDest, 2),
        'newbalanceDest': np.round(newbalDest, 2),
        'isFraud': 1,
    })

df = pd.concat([gen_genuine(n_genuine), gen_fraud(n_fraud)], ignore_index=True).sample(frac=1, random_state=42)
print(f'Total: {len(df):,}, Fraud: {df["isFraud"].sum():,} ({df["isFraud"].mean()*100:.3f}%)')

## 2. Feature Engineering

In [3]:
def add_features(df):
    df = df.copy()
    df['balance_diff_orig'] = df['newbalanceOrig'] - df['oldbalanceOrg'] + df['amount']
    df['balance_diff_dest'] = df['newbalanceDest'] - df['oldbalanceDest'] - df['amount']
    df['orig_drained'] = (df['newbalanceOrig'] == 0).astype(int)
    df['dest_unchanged'] = (df['newbalanceDest'] == df['oldbalanceDest']).astype(int)
    df['amount_to_balance'] = df['amount'] / (df['oldbalanceOrg'] + 1)
    return df

df = add_features(df)
FEATURE_COLS = [
    'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest',
    'balance_diff_orig', 'balance_diff_dest', 'orig_drained', 'dest_unchanged', 'amount_to_balance'
]
TARGET = 'isFraud'
X = df[FEATURE_COLS]
y = df[TARGET]

## 3. Train-Test Split and Preprocessing Pipeline

In [4]:
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=42)

cat_cols = ['type']
num_cols = [c for c in FEATURE_COLS if c not in cat_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ],
    remainder='drop',
)
X_train_pre = preprocessor.fit_transform(X_train)
X_val_pre = preprocessor.transform(X_val)
X_test_pre = preprocessor.transform(X_test)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_pre, y_train)

## 4. Model Training & Threshold Calibration

In [5]:
model = XGBClassifier(
    n_estimators=400, max_depth=7, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, eval_metric='aucpr', random_state=42, n_jobs=-1
)
model.fit(X_train_res, y_train_res, eval_set=[(X_val_pre, y_val)], verbose=False)

val_proba = model.predict_proba(X_val_pre)[:, 1]
best_f1, best_thr = 0, 0.5
for thr in np.arange(0.05, 0.95, 0.01):
    preds = (val_proba >= thr).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr
print(f'Optimal Threshold: {best_thr:.2f} (Val F1: {best_f1:.4f})')

## 5. Test Set Evaluation

In [6]:
test_proba = model.predict_proba(X_test_pre)[:, 1]
test_preds = (test_proba >= best_thr).astype(int)
print('Accuracy: ', accuracy_score(y_test, test_preds))
print('Precision:', precision_score(y_test, test_preds))
print('Recall:   ', recall_score(y_test, test_preds))
print('F1 Score: ', f1_score(y_test, test_preds))
print('ROC-AUC:  ', roc_auc_score(y_test, test_proba))
print('PR-AUC:   ', average_precision_score(y_test, test_proba))
print('Confusion Matrix:\n', confusion_matrix(y_test, test_preds))